# HYDROSHIP — Klasifikasi Sisi Kolam dari QR Payload Bawah Air

Melatih CNN 4-kelas (**A/B/C/D**) pada dataset simulasi bawah air yang dihasilkan
`autonomy/tests/evaluate_qr_underwater.py`.

## Kenapa klasifikasi sisi kolam, bukan "readable / not readable"?

Classifier "bisa dibaca / tidak" hanya memberi tahu ROV bahwa ia **gagal** — tak menolong misi.
Sebaliknya, menentukan **sisi kolam** hanya perlu membedakan **4 pola tetap**, jauh lebih mudah
daripada mendekode data arbitrer. Jadi model tetap mengenali sisi kolam **justru di frame tempat
`decode_qr()` menyerah** → memperluas jangkauan operasi ROV.

## ✅ VERDICT — sudah diukur, CNN MENANG

Diuji pada dataset `--seed 1` (realisasi riak sama sekali berbeda, belum pernah dilihat model),
subset `decode_qr()` GAGAL:

| misalignment | **CNN** | template matching | catatan |
|---|---|---|---|
| ±0 px ±0° | 100% | 100% | crop sempurna — tak realistis |
| ±5 px ±6° | **100%** | 82.2% | dalam distribusi latih |
| ±10 px ±12° | **100%** | 53.9% | dalam distribusi latih |
| ±20 px ±25° | **100%** | 25.0% (=acak) | **di luar** distribusi latih |
| ±30 px ±40° | **85.5%** | 28.3% | **di luar** distribusi latih |
| ±50 px ±60° | **63.8%** | 33.6% | **di luar** distribusi latih |

**Cakupan: `decode_qr()` 78.9% → +CNN = 100%.** CNN menutup SELURUH kegagalan decode dan
terdegradasi mulus jauh di luar distribusi latihnya.

Kenapa template matching runtuh: ia butuh QR ter-crop sempurna. Di rekaman ROV asli QR tak
pernah pas di tengah — dan melokalisasinya justru sulit tepat ketika decode gagal. **Di situlah
CNN menang: ketahanan misalignment**, bukan akurasi mentah.

## ⚠️ Training BISA gagal diam-diam — jangan lewati sanity check

Pernah terjadi sungguhan: checkpoint dgn val_acc ~33% (tebak acak 25%) yang **tak pernah
menebak A atau B** — kolaps ke 2 kelas, dan **salah pada 2 dari 4 template BERSIH**. Loss tetap
terlihat menurun. Arsitektur di notebook ini **benar** (terbukti mencapai 100% val di epoch ~4,
±1 menit bahkan di CPU) — jadi kalau hasil Anda buruk, **itu run-nya yang gagal, bukan
desainnya: cukup latih ulang.** Sel sanity check sesudah training akan menangkapnya.

## Fakta terukur dari sweep robustness

| Temuan | Implikasi |
|---|---|
| **Riak/caustics = 100% penyebab gagal decode.** Efek fotometrik (warna/kabut/blur) saja tak pernah mematahkan pyzbar, bahkan di depth=1.0 | Model harus tahan **distorsi geometris**, bukan sekadar kontras |
| Tebing decode ≈ **0.4 lebar modul** perpindahan riak | Toleransi berskala dgn ukuran QR di frame |
| **Sisi D & C lebih rapuh** dari A & B (pola modul beda krn masking QR) | D gagal decode ~2× lebih sering dari B |
| QR cetakan = `HYDROSHIP-M5-A` (string biasa, **bukan JSON**), versi-1 | 4 template tetap → tugas klasifikasi wajar |

⚠️ **Batas simulasi — kenapa 100% itu optimistis:** train & test memakai **4 template bersih
yang sama** (hanya degradasinya beda), jadi model menghafal 4 pola lalu mengenalinya di bawah
noise. QR cetakan yang difoto sungguhan akan berbeda. Simulasi juga tak memodelkan turbiditas
nyata, glare permukaan, motion blur, maupun **perspektif** (rotasi hanya disuntik lewat
`MISALIGN_*`). **Validasi dgn rekaman kolam sebelum dipasang ke ROV.**

## 1. Mount Drive & extract dataset

Upload dulu `autonomy/dataset.zip` ke Drive, mis. ke `MyDrive/qr_underwater/dataset.zip`.

Regenerate dataset (di mesin lokal, dari dalam `autonomy/`) — `--samples 84` → **3.024 gambar**
(rumus: `36 × samples`):
```bash
python tests/evaluate_qr_underwater.py --ids A,B,C,D --samples 84 --seed 0 --outdir dataset --zip
```

Untuk uji generalisasi yang jujur, buat juga set kedua dgn seed berbeda:
```bash
python tests/evaluate_qr_underwater.py --ids A,B,C,D --samples 20 --seed 1 --outdir dataset_seed1 --zip
```

In [ ]:
import os, zipfile

ZIP_PATH = '/content/drive/MyDrive/qr_underwater/dataset.zip'  # ← sesuaikan
DATA_DIR = '/content/dataset'

from google.colab import drive
drive.mount('/content/drive')

assert os.path.exists(ZIP_PATH), f'zip tak ditemukan: {ZIP_PATH}'
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content')

print('isi dataset:', sorted(os.listdir(DATA_DIR)))

In [ ]:
import pandas as pd

df = pd.read_csv(os.path.join(DATA_DIR, 'metadata.csv'))
df['abspath'] = df['path'].apply(lambda p: os.path.join(DATA_DIR, p))

assert df['abspath'].map(os.path.exists).all(), 'ada path gambar yang hilang'

print(f'total gambar        : {len(df)}')
print(f'label (qr_id)       : {df["qr_id"].value_counts().to_dict()}')
print(f'depth_level         : {df["depth_level"].value_counts().to_dict()}')
print(f'payload groundtruth : {sorted(df["payload_groundtruth"].unique())}')

# BASELINE yang harus dikalahkan — decode_qr() klasik, tanpa training sama sekali.
cov = df['decode_enhance_ok'].mean()
n_fail = int((~df['decode_enhance_ok']).sum())
print(f'\ncakupan decode_qr()  : {cov:.1%}  ({n_fail} frame GAGAL ← target model)')
print('decode gagal per label :', df[~df['decode_enhance_ok']]['qr_id'].value_counts().to_dict())
print('decode gagal per level :', df[~df['decode_enhance_ok']]['depth_level'].value_counts().to_dict())
df.head(3)

## 2. Lihat datanya dulu

Selalu pandangi gambar sebelum melatih — kalau `deep` sudah tak terbaca mata, jangan
berharap model ajaib.

In [ ]:
import cv2
import matplotlib.pyplot as plt

levels = ['shallow', 'medium', 'deep']
fig, axes = plt.subplots(4, len(levels), figsize=(3 * len(levels), 12))
for r, wall in enumerate(['A', 'B', 'C', 'D']):
    for c, lvl in enumerate(levels):
        sub = df[(df.qr_id == wall) & (df.depth_level == lvl) & (df.module_px == 8)]
        ax = axes[r, c]
        if len(sub):
            row = sub.iloc[0]
            img = cv2.cvtColor(cv2.imread(row['abspath']), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ok = 'decode OK' if row['decode_enhance_ok'] else 'decode GAGAL'
            ax.set_title(f'{wall} · {lvl}\n{ok}', fontsize=9)
        ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Preprocessing + simulasi misalignment

**Crop ternormalisasi skala.** QR selalu di tengah kanvas dan `qr_px` merekam lebarnya,
jadi crop tepat di QR lalu resize — jarak (`module_px`) tak lagi jadi variabel pengganggu.

**Grayscale, bukan warna.** Atenuasi warna berkorelasi kuat dgn `depth_level` tapi
**tak membawa informasi sisi kolam** — memakai warna hanya mengundang model belajar
pintasan yang salah (menebak depth, bukan wall).

**`MISALIGN_PX` / `MISALIGN_DEG`** mensimulasikan lokalisasi QR yang tak sempurna di
lapangan. Inilah kondisi uji yang sesungguhnya — set 0 dan template matching menang telak
tanpa training.

In [ ]:
import numpy as np, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG = 96
WALLS = ['A', 'B', 'C', 'D']
W2I = {w: i for i, w in enumerate(WALLS)}
CANVAS_W, CANVAS_H = 640, 480   # kanvas default generator

# Seberapa buruk lokalisasi QR di lapangan. NAIKKAN utk uji yang lebih realistis.
MISALIGN_PX = 10
MISALIGN_DEG = 12
print('device:', DEV, '| misalign: ±%dpx ±%d°' % (MISALIGN_PX, MISALIGN_DEG))


def load_gray(path):
    return cv2.imread(path, cv2.IMREAD_GRAYSCALE)


def make_crop(img, qr_px, dx=0, dy=0, deg=0.0, margin=1.15):
    """Crop kotak di tengah seluas QR (+margin), dgn geser/rotasi opsional → skala normal."""
    if deg:
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), deg, 1.0)
        img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_REFLECT)
    side = int(qr_px * margin)
    cx, cy = CANVAS_W // 2 + int(dx), CANVAS_H // 2 + int(dy)
    x0 = max(0, cx - side // 2); y0 = max(0, cy - side // 2)
    crop = img[y0:y0 + side, x0:x0 + side]
    crop = cv2.resize(crop, (IMG, IMG), interpolation=cv2.INTER_AREA).astype(np.float32)
    # normalisasi per-gambar: buang bias kecerahan/kontras global (efek backscatter)
    return (crop - crop.mean()) / (crop.std() + 1e-6)


class QRWallDataset(Dataset):
    """train=True → augmentasi geser+rotasi (inilah yg membuat CNN unggul atas template
    matching); train=False → misalignment TETAP per-indeks (seed dari idx) supaya
    perbandingan CNN vs template adil di kondisi yang sama persis."""

    def __init__(self, frame, train=False, misalign=True):
        self.df = frame.reset_index(drop=True)
        self.train = train
        self.misalign = misalign

    def __len__(self):
        return len(self.df)

    def _jitter(self, i):
        if not self.misalign:
            return 0, 0, 0.0
        rng = np.random.default_rng(None if self.train else [SEED, i])
        return (rng.integers(-MISALIGN_PX, MISALIGN_PX + 1),
                rng.integers(-MISALIGN_PX, MISALIGN_PX + 1),
                float(rng.uniform(-MISALIGN_DEG, MISALIGN_DEG)))

    def __getitem__(self, i):
        r = self.df.iloc[i]
        dx, dy, deg = self._jitter(i)
        x = make_crop(load_gray(r['abspath']), r['qr_px'], dx, dy, deg)
        if self.train:
            x = x + np.random.normal(0, 0.05, x.shape).astype(np.float32)
        return torch.from_numpy(x).unsqueeze(0).float(), W2I[r['qr_id']]


strat = df['qr_id'] + '_' + df['depth_level']
train_df, val_df = train_test_split(df, test_size=0.25, random_state=SEED, stratify=strat)

train_dl = DataLoader(QRWallDataset(train_df, train=True), batch_size=64, shuffle=True, num_workers=2)
val_dl = DataLoader(QRWallDataset(val_df), batch_size=64, num_workers=2)
print(f'train={len(train_df)}  val={len(val_df)}')
print('val decode GAGAL:', int((~val_df['decode_enhance_ok']).sum()), '← subset penentu')

## 4. Baseline: template matching (tanpa training)

Inilah yang harus dikalahkan CNN. Diuji pada **misalignment yang sama** dgn val set CNN.

In [ ]:
# 4 template dari gambar bersih
TMPL = {}
for w in WALLS:
    row = df[(df.qr_id == w) & (df.module_px == 8)].iloc[0]
    p = os.path.join(DATA_DIR, 'clean', f'qr_{w}_m8_clean.png')
    TMPL[w] = make_crop(load_gray(p), row['qr_px'])


def tmpl_predict(crop):
    return max(WALLS, key=lambda w: float((crop * TMPL[w]).mean()))


def tmpl_eval(frame, misalign=True):
    ds = QRWallDataset(frame, train=False, misalign=misalign)
    ok = []
    for i in range(len(ds)):
        r = ds.df.iloc[i]
        dx, dy, deg = ds._jitter(i)
        c = make_crop(load_gray(r['abspath']), r['qr_px'], dx, dy, deg)
        ok.append(tmpl_predict(c) == r['qr_id'])
    return np.array(ok)


val_fail = val_df[~val_df['decode_enhance_ok']]
tm_aligned = tmpl_eval(val_fail, misalign=False)
tm_mis = tmpl_eval(val_fail, misalign=True)
print(f'TEMPLATE MATCHING pada val subset decode-GAGAL (n={len(val_fail)}):')
print(f'  crop sempurna         : {tm_aligned.mean():.1%}   ← tak realistis')
print(f'  ★ misalign ±{MISALIGN_PX}px ±{MISALIGN_DEG}° : {tm_mis.mean():.1%}   ← ANGKA YANG HARUS DIKALAHKAN CNN')

## 5. Model

CNN kecil sengaja dipilih: hanya 4 kelas dari 4 template tetap. Model besar (ResNet dll.)
akan menghafal ~2 ribu gambar ini, bukan belajar ketahanan.

In [ ]:
class WallCNN(nn.Module):
    def __init__(self, n_cls=4):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o),
                                 nn.ReLU(), nn.MaxPool2d(2))
        self.f = nn.Sequential(blk(1, 16), blk(16, 32), blk(32, 64), blk(64, 64))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(64, n_cls))

    def forward(self, x):
        return self.head(self.f(x))


model = WallCNN().to(DEV)
print('parameter:', sum(p.numel() for p in model.parameters()))

In [ ]:
EPOCHS = 30
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
crit = nn.CrossEntropyLoss()


def evaluate(dl):
    model.eval(); correct = tot = 0
    with torch.no_grad():
        for x, y in dl:
            p = model(x.to(DEV)).argmax(1).cpu()
            correct += (p == y).sum().item(); tot += len(y)
    return correct / tot


best = 0.0
for ep in range(1, EPOCHS + 1):
    model.train(); run = 0.0
    for x, y in train_dl:
        opt.zero_grad()
        loss = crit(model(x.to(DEV)), y.to(DEV))
        loss.backward(); opt.step()
        run += loss.item() * len(y)
    sched.step()
    acc = evaluate(val_dl)
    if acc > best:
        best = acc; torch.save(model.state_dict(), '/content/best.pt')
    if ep % 5 == 0 or ep == 1:
        print(f'ep {ep:>3}  loss={run/len(train_df):.4f}  val_acc={acc:.3f}  best={best:.3f}')

model.load_state_dict(torch.load('/content/best.pt'))
print('\nval_acc terbaik:', round(best, 4))

In [ ]:
# ── SANITY CHECK — JANGAN LEWATI ──────────────────────────────────────────────
# Training BISA gagal diam-diam: model kolaps ke 1-2 kelas & tetap melaporkan loss
# yang menurun. Ini pernah terjadi sungguhan (val_acc ~33%, tak pernah menebak A/B).
# Dua uji murah yang menangkapnya seketika:
#   1. Model HARUS benar pada 4 template BERSIH (input termudah yang mungkin ada).
#   2. Prediksi HARUS memakai keempat kelas — bukan kolaps ke sebagian.
# Kalau gagal: JANGAN dipakai. Latih ulang (biasanya cukup ulang dgn seed beda).
model.eval()
print('1) prediksi pada 4 template BERSIH (tanpa degradasi, crop sempurna):')
clean_ok = True
for w in WALLS:
    row = df[(df.qr_id == w) & (df.module_px == 8)].iloc[0]
    c = make_crop(load_gray(os.path.join(DATA_DIR, 'clean', f'qr_{w}_m8_clean.png')), row['qr_px'])
    with torch.no_grad():
        logit = model(torch.from_numpy(c)[None, None].float().to(DEV))
        prob = torch.softmax(logit, 1)[0].cpu()
    pred = WALLS[int(logit.argmax(1))]
    mark = '✓' if pred == w else '✗'
    clean_ok &= (pred == w)
    print(f'   [{mark}] asli={w} -> {pred}   probs=' + ' '.join(f'{x:.2f}' for x in prob))

import collections
dist = collections.Counter(WALLS[p] for p in preds) if 'preds' in dir() else None
if dist is None:
    _p = []
    with torch.no_grad():
        for x, _y in val_dl:
            _p += model(x.to(DEV)).argmax(1).cpu().tolist()
    dist = collections.Counter(WALLS[i] for i in _p)
print(f'\n2) distribusi prediksi di val: {dict(dist)}')
used = len(dist)

print()
if not clean_ok:
    raise RuntimeError('SANITY GAGAL: model salah pada template BERSIH → training kolaps. '
                       'JANGAN pakai checkpoint ini; latih ulang.')
if used < len(WALLS):
    raise RuntimeError(f'SANITY GAGAL: model hanya memakai {used}/4 kelas → kolaps. Latih ulang.')
print('✓ SANITY LULUS — model memakai keempat kelas & benar pada semua template bersih.')

## 6. Evaluasi — apakah CNN benar-benar menambah nilai?

Dua pertanyaan, dijawab pada **misalignment yang sama** untuk kedua metode:

1. Di frame yang `decode_qr()` gagal, seberapa sering model benar? (baseline acak = 25%)
2. Apakah CNN **mengalahkan template matching**? Kalau tidak → buang CNN-nya.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
preds, trues = [], []
with torch.no_grad():
    for x, y in DataLoader(QRWallDataset(val_df), batch_size=64):
        preds += model(x.to(DEV)).argmax(1).cpu().tolist(); trues += y.tolist()

res = val_df.reset_index(drop=True).copy()
res['pred'] = [WALLS[p] for p in preds]
res['benar'] = res['pred'] == res['qr_id']
gagal = res[~res['decode_enhance_ok']]
bisa = res[res['decode_enhance_ok']]

print('=' * 66)
print(f' CNN vs TEMPLATE MATCHING vs decode_qr()   [misalign ±{MISALIGN_PX}px ±{MISALIGN_DEG}°]')
print('=' * 66)
print(f"  akurasi CNN keseluruhan        : {res['benar'].mean():.1%}   (menyesatkan!)")
print(f"  akurasi CNN saat decode OK     : {bisa['benar'].mean():.1%}   (n={len(bisa)}) — decode_qr sudah 100% di sini")
if len(gagal):
    cnn_f = gagal['benar'].mean()
    print(f"\n  ── subset decode-GAGAL (n={len(gagal)}) ── ANGKA PENENTU")
    print(f"    tebak acak                   : 25.0%")
    print(f"    template matching            : {tm_mis.mean():.1%}")
    print(f"    ★ CNN                        : {cnn_f:.1%}")
    delta = cnn_f - tm_mis.mean()
    verdict = ('CNN MENANG → layak dipakai' if delta > 0.02 else
               'CNN TAK LEBIH BAIK → pakai template matching (tanpa GPU/training)')
    print(f"    selisih                      : {delta:+.1%}  → {verdict}")
    print(f"\n  cakupan decode_qr saja         : {res['decode_enhance_ok'].mean():.1%}")
    print(f"  cakupan decode_qr + CNN        : {(res['decode_enhance_ok'].sum() + gagal['benar'].sum())/len(res):.1%}  ← perluasan jangkauan")
    print('\n  per level (subset decode GAGAL):')
    for lvl, g in gagal.groupby('depth_level'):
        print(f"    {lvl:<8}: {g['benar'].mean():.1%}  (n={len(g)})")
else:
    print('  (tak ada frame gagal di val — perbesar --samples)')
print('=' * 66)

print('\n', classification_report(trues, preds, target_names=WALLS, digits=3))
print('confusion matrix (baris=asli, kolom=prediksi):')
print(pd.DataFrame(confusion_matrix(trues, preds), index=WALLS, columns=WALLS))

In [ ]:
# Simpan model + label mapping ke Drive
OUT = '/content/drive/MyDrive/qr_underwater'
os.makedirs(OUT, exist_ok=True)
torch.save({'state_dict': model.state_dict(), 'walls': WALLS, 'img_size': IMG,
            'canvas': [CANVAS_W, CANVAS_H]}, f'{OUT}/wall_cnn.pt')
print('tersimpan:', f'{OUT}/wall_cnn.pt')

## 7. Langkah berikutnya

**Kalau CNN mengalahkan template matching** pada subset decode-GAGAL: layak jadi *fallback*
sesudah `decode_qr()` mengembalikan `[]` di `VisionPipeline`. Catatan penting: model hanya
memberi **wall**, bukan payload — `_is_target_payload()` tak bisa memvalidasinya, jadi
perlakukan sbg petunjuk berkeyakinan rendah (mis. butuh N frame konsisten sebelum dipercaya).

**Kalau tidak menang:** pakai template matching — 20 baris, tanpa GPU. Jangan pasang CNN
hanya karena sudah terlanjur dilatih.

**Masalah sebenarnya mungkin bukan klasifikasi, tapi LOKALISASI.** Kedua metode butuh tahu
QR ada di mana; template matching runtuh ke 25% pada geser ±20px. Kalau QR bisa dilokalisasi
akurat, decode sering sudah berhasil duluan. Jadi pertimbangkan dulu yang lebih berdampak:
- **rata-rata multi-frame** untuk meredam riak (riak berosilasi, QR diam) ← paling menjanjikan,
- dekatkan ROV / perbesar QR cetak (tebing ada di ~0.4 lebar modul),
- payload lebih pendek lagi → versi QR lebih rendah → modul lebih besar.

**Uji generalisasi yang jujur:** latih pakai `--seed 0`, uji pakai dataset `--seed 1`
(realisasi riak/jitter yang sama sekali berbeda). Val split di notebook ini masih memakai
distribusi degradasi yang sama.

**Sebelum percaya angka mana pun:** simulasi ini tak punya rotasi/perspektif bawaan
(hanya disuntik lewat `MISALIGN_*` di sini), tak punya turbiditas nyata, glare, atau motion
blur. Validasi dgn rekaman kolam sungguhan sebelum dipasang ke ROV.